# Deep Dive into Rozans 618

This notebook focuses entirely on YOUR data -- the 618 peptides from your three papers. We'll cross-reference against MEROPS, Turk, and ESM-2 embeddings to see where external data agrees or disagrees with your experiments.

**Goal:** Identify which features and data sources are actually predictive of what you observed, and where the gaps are that your 80K dataset will fill.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.exists("data"):
    # Try parent directory
    os.chdir(os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import pickle

%matplotlib inline
import matplotlib.pyplot as plt

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

from exopred.data_pipeline import _three_to_one

print(f"Working directory: {os.getcwd()}")

---
## 1. Load and Split by Paper

In [ ]:
rozans = pd.read_csv("data/rozans-618-enriched.csv")

paper1 = rozans[rozans["paper"] == "Paper 1 (ACS Biomater 2024)"].copy()
paper2 = rozans[rozans["paper"] == "Paper 2 (JBMR-A 2025)"].copy()
paper3 = rozans[rozans["paper"] == "Paper 3 (Adv Healthcare Mater 2025)"].copy()

print(f"Paper 1 (ACS Biomater 2024) -- Adhesion peptides with terminal mods: {len(paper1)} peptides")
print(f"  Libraries: {paper1['library'].nunique()}")
print(f"  Unique sequences: {paper1['clean_sequence'].nunique()}")
print(f"  N-terminal mods: {sorted(paper1['n_terminal'].unique())}")
print(f"  C-terminal mods: {sorted(paper1['c_terminal'].unique())}")

print(f"\nPaper 2 (JBMR-A 2025) -- Bone-related: {len(paper2)} peptides")
print(f"  Types: {paper2['type'].value_counts().to_dict()}")

print(f"\nPaper 3 (Adv Healthcare Mater 2025) -- MMP-14 crosslinkers: {len(paper3)} peptides")
print(f"  Libraries: {paper3['library'].nunique()}")
print(f"  Unique sequences: {paper3['clean_sequence'].nunique()}")

---
## 2. Paper 1: Aminopeptidase vs Carboxypeptidase Susceptibility

These susceptibility scores are **computed from MEROPS lookup tables**, not from your measurements. Let's see if they make sense.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by N-terminal modification
n_mod_colors = {
    "NH2": "#e53935", "Ac": "#1e88e5", "Ac-\u03b2A": "#43a047",
    "NH2-\u03b2A": "#fb8c00", "N-\u03b2A": "#8e24aa",
    "cyclo": "#00acc1", "N3": "#6d4c41", "NH2-\u03b2F": "#546e7a",
}

for n_mod in paper1["n_terminal"].unique():
    sub = paper1[paper1["n_terminal"] == n_mod]
    color = n_mod_colors.get(n_mod, "#999999")
    axes[0].scatter(sub["aminopeptidase_susceptibility"], 
                    sub["carboxypeptidase_susceptibility"],
                    label=n_mod, alpha=0.6, s=40, color=color)

axes[0].set_xlabel("Aminopeptidase Susceptibility (computed)")
axes[0].set_ylabel("Carboxypeptidase Susceptibility (computed)")
axes[0].set_title("Paper 1: Exopeptidase Susceptibility\n(colored by N-terminal modification)")
axes[0].legend(fontsize=8, loc="upper right")

# Same plot but colored by C-terminal modification
c_mod_colors = {"COOH": "#e53935", "amide": "#1e88e5", "\u03b2A": "#43a047"}

for c_mod in paper1["c_terminal"].unique():
    sub = paper1[paper1["c_terminal"] == c_mod]
    color = c_mod_colors.get(c_mod, "#999999")
    axes[1].scatter(sub["aminopeptidase_susceptibility"], 
                    sub["carboxypeptidase_susceptibility"],
                    label=c_mod, alpha=0.6, s=40, color=color)

axes[1].set_xlabel("Aminopeptidase Susceptibility (computed)")
axes[1].set_ylabel("Carboxypeptidase Susceptibility (computed)")
axes[1].set_title("Paper 1: Exopeptidase Susceptibility\n(colored by C-terminal modification)")
axes[1].legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

# Quick stats
print("Aminopeptidase susceptibility by N-terminal mod:")
print(paper1.groupby("n_terminal")["aminopeptidase_susceptibility"].describe()[["mean", "std", "min", "max"]].round(3).to_string())
print("\nCarboxypeptidase susceptibility by C-terminal mod:")
print(paper1.groupby("c_terminal")["carboxypeptidase_susceptibility"].describe()[["mean", "std", "min", "max"]].round(3).to_string())

### SAM'S EXPERTISE NEEDED

These susceptibility scores are computed from MEROPS lookup tables, not your measurements. **How well do they correlate with what you actually observed?**

- Did Ac- (acetylated) peptides consistently show lower degradation in your LC-MS data?
- Did betaA C-terminal peptides outperform COOH and amide as the scores suggest?
- The scores only capture the TERMINAL residue effect. Your data showed the VARIABLE residue also matters (e.g., Histidine). How much signal is the model missing?

Correlation estimate (your gut): ___% of the variance explained by terminal mods alone.

---
## 3. Terminal Modification Coverage Matrix

Which (N-mod, C-mod) combinations are represented? Where are the gaps?

In [ ]:
# Coverage matrix for Paper 1 (where terminal mods matter most)
coverage = pd.crosstab(paper1["n_terminal"], paper1["c_terminal"], margins=True)
print("Paper 1: Terminal Modification Coverage (peptide count)")
print(coverage.to_string())

print(f"\nTotal unique (n_mod, c_mod) combos: {paper1.groupby(['n_terminal', 'c_terminal']).ngroups}")

# Are any combos missing?
all_n = sorted(paper1["n_terminal"].unique())
all_c = sorted(paper1["c_terminal"].unique())
print(f"\nAll N-mods: {all_n}")
print(f"All C-mods: {all_c}")
print(f"Possible combos: {len(all_n) * len(all_c)}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
coverage_no_margin = pd.crosstab(paper1["n_terminal"], paper1["c_terminal"])
if HAS_SEABORN:
    sns.heatmap(coverage_no_margin, annot=True, fmt="d", cmap="Blues", ax=ax,
                linewidths=0.5, cbar_kws={"label": "Peptide Count"})
else:
    im = ax.imshow(coverage_no_margin.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(coverage_no_margin.columns)))
    ax.set_xticklabels(coverage_no_margin.columns, rotation=45)
    ax.set_yticks(range(len(coverage_no_margin.index)))
    ax.set_yticklabels(coverage_no_margin.index)
    for i in range(len(coverage_no_margin.index)):
        for j in range(len(coverage_no_margin.columns)):
            ax.text(j, i, str(coverage_no_margin.iloc[i, j]), ha="center", va="center")
    plt.colorbar(im, ax=ax)

ax.set_title("Terminal Modification Coverage (Paper 1)")
plt.tight_layout()
plt.show()

# Identify gaps
missing = []
for n in all_n:
    for c in all_c:
        count = len(paper1[(paper1["n_terminal"] == n) & (paper1["c_terminal"] == c)])
        if count == 0:
            missing.append((n, c))
if missing:
    print(f"\nMISSING combinations ({len(missing)}):")
    for n, c in missing:
        print(f"  {n} + {c}")
else:
    print("\nAll combinations represented!")

---
## 4. Variable Residue Analysis -- The Histidine Mystery

Paper 1 uses the RGEFV-X scaffold where X is one of 19 standard amino acids plus the scaffold alone. The key finding: **terminal modifications dominate, but Histidine was the exception** -- it degraded even when acetylated.

In [ ]:
# Focus on libraries with variable residues (exclude named/internal/additional)
var_libs = paper1[paper1["variable_residue"].notna() & (paper1["variable_residue"] != "")].copy()
print(f"Peptides with variable residue: {len(var_libs)}")
print(f"Unique variable AAs: {sorted(var_libs['variable_residue'].unique())}")

# For each variable AA, show computed properties
aa_props = var_libs.groupby("variable_residue").agg(
    count=("sequence", "count"),
    mean_aminopep=("aminopeptidase_susceptibility", "mean"),
    mean_carboxypep=("carboxypeptidase_susceptibility", "mean"),
    mean_total_exo=("total_exopeptidase_susceptibility", "mean"),
    mean_gravy=("gravy", "mean"),
    mean_instability=("instability_index", "mean"),
    mean_mw=("mw_da", "mean"),
).round(3)

print("\nComputed properties by variable residue:")
print(aa_props.sort_values("mean_total_exo", ascending=False).to_string())

# Highlight Histidine
if "H" in var_libs["variable_residue"].values:
    h_peps = var_libs[var_libs["variable_residue"] == "H"]
    print(f"\n{'='*60}")
    print(f"HISTIDINE peptides ({len(h_peps)}):")
    print(h_peps[["sequence", "n_terminal", "c_terminal", 
                   "aminopeptidase_susceptibility", "carboxypeptidase_susceptibility",
                   "total_exopeptidase_susceptibility"]].to_string(index=False))

# Plot: total exopeptidase susceptibility by variable AA, grouped by protection level
fig, ax = plt.subplots(figsize=(12, 6))

# Group by (variable_residue, n_terminal protection)
var_libs["protection"] = var_libs["n_terminal"].map({
    "NH2": "Unprotected", "Ac": "Acetylated", "Ac-\u03b2A": "Acetylated+\u03b2A",
    "NH2-\u03b2A": "\u03b2A only",
}).fillna("Other")

for prot, color in [("Unprotected", "#e53935"), ("Acetylated", "#1e88e5"), 
                     ("Acetylated+\u03b2A", "#43a047"), ("\u03b2A only", "#fb8c00")]:
    sub = var_libs[var_libs["protection"] == prot]
    if len(sub) == 0:
        continue
    aa_means = sub.groupby("variable_residue")["total_exopeptidase_susceptibility"].mean()
    ax.scatter(aa_means.index, aa_means.values, label=prot, s=60, alpha=0.7, color=color)

ax.set_xlabel("Variable Residue (X in RGEFV-X)")
ax.set_ylabel("Mean Total Exopeptidase Susceptibility")
ax.set_title("How Does the Variable Residue Affect Susceptibility?")
ax.legend()
ax.axhline(var_libs["total_exopeptidase_susceptibility"].mean(), color="gray", 
           linestyle="--", alpha=0.5, label="Overall mean")
plt.tight_layout()
plt.show()

### SAM'S EXPERTISE NEEDED

**Histidine was the exception -- degraded even when acetylated. Can you think of why?**

Possible hypotheses:
1. **Charge at physiological pH** -- His has pKa ~6.0, so it's partially protonated at pH 7.4. Could the positive charge recruit endopeptidases?
2. **Imidazole ring** -- Could the ring act as a metal coordination site, attracting metalloprotease active sites?
3. **Conformational effect** -- Does His change the backbone flexibility in a way that exposes internal cleavage sites?
4. **Something else entirely?**

Your hypothesis:
- Most likely explanation: 
- How to test it: 

---
## 5. Paper 3: MMP-14 Crosslinkers vs Turk Z-Scores

Your Paper 3 has 364 crosslinker peptides from a split-and-pool library. Let's cross-reference them against the Turk 2015 MMP-14 Z-scores.

In [ ]:
turk = pd.read_excel("data/turk2015/mmc2-table-S1.xlsx")
turk = turk.rename(columns={"Unnamed: 0": "sequence"})

print(f"Paper 3 crosslinkers: {len(paper3)} peptides")
print(f"Turk dataset: {len(turk):,} peptides")

# Extract subsequences from Paper 3 to match against Turk
# Paper 3 sequences are crosslinkers -- they may contain the MMP cleavage motif
# Let's check if any Paper 3 sequences appear as subsequences in Turk

# First, look at the computed MMP scores already in the enriched data
print(f"\nPaper 3 MMP scores (pre-computed):")
print(paper3[["mmp_p1_score", "mmp_p1p_score", "mmp_cleavage_score"]].describe().round(3).to_string())

# Compare P1 residue distributions between Paper 3 and high-scoring Turk peptides
turk["P1_aa"] = turk["sequence"].str[4]

# For Paper 3, what would the P1 position be? Depends on where MMP-14 cuts.
# The mmp_p1_score in the enriched data tells us the score of the P1 position.
# Let's look at the actual sequences
print(f"\nSample Paper 3 sequences:")
print(paper3[["clean_sequence", "mmp_p1_score", "mmp_cleavage_score"]].head(10).to_string(index=False))

# Distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Paper 3 mmp_cleavage_score distribution
axes[0].hist(paper3["mmp_cleavage_score"].dropna(), bins=30, color="steelblue", 
             edgecolor="white", alpha=0.8)
axes[0].set_xlabel("MMP Cleavage Score (computed)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Paper 3 Crosslinkers: MMP Cleavage Scores (n={len(paper3)})")

# Turk MMP-14 Z-score for comparison
turk_top = turk.nlargest(364, "MMP14")  # Match Paper 3 size
axes[1].hist(turk["MMP14"], bins=80, color="gray", edgecolor="white", alpha=0.4, label="All Turk")
axes[1].hist(turk_top["MMP14"], bins=30, color="#d32f2f", edgecolor="white", alpha=0.6, 
             label=f"Top {len(paper3)} Turk")
axes[1].set_xlabel("MMP-14 Z-score (Turk)")
axes[1].set_ylabel("Count")
axes[1].set_title("Turk MMP-14: All vs Top 364")
axes[1].legend()

plt.tight_layout()
plt.show()

# P1 amino acid comparison
p3_c_term = paper3["c_term_aa"].value_counts().head(10)
turk_top_p1 = turk_top["P1_aa"].value_counts().head(10)
print(f"\nP1 residue comparison:")
print(f"{'AA':<5} {'Paper3 C-term freq':<20} {'Turk top-364 P1 freq':<20}")
for aa in set(list(p3_c_term.index) + list(turk_top_p1.index)):
    p3_freq = p3_c_term.get(aa, 0) / len(paper3)
    turk_freq = turk_top_p1.get(aa, 0) / len(turk_top)
    print(f"{aa:<5} {p3_freq:<20.3f} {turk_freq:<20.3f}")

### SAM'S EXPERTISE NEEDED

Do the sequences that scored high in Turk's purified MMP-14 assay overlap with the ones that **actually worked** in your split-and-pool screen?

- The Turk data is purified enzyme + short peptide substrate. Your system is cells in a matrix.
- If there's good agreement, Turk Z-scores are a powerful feature for the model.
- If there's disagreement, it tells us the cellular context matters more than the enzyme specificity alone.

Agreement level (your assessment): 

---
## 6. MEROPS Cross-Reference for Paper 1 Sequences

For each of Sam's 24 unique Paper 1 sequences (19 variable + scaffold + named), look up the terminal amino acids in the MEROPS exopeptidase data.

In [ ]:
merops = pd.read_csv("data/processed/merops_exopeptidase_cleavages.csv")

# Convert MEROPS P1 to 1-letter
merops["P1_1"] = merops["P1"].apply(
    lambda x: _three_to_one(str(x).strip()) if pd.notna(x) else None
)
merops["P1p_1"] = merops["P1prime"].apply(
    lambda x: _three_to_one(str(x).strip()) if pd.notna(x) else None
)

# Get unique Paper 1 sequences
p1_seqs = paper1["clean_sequence"].unique()
print(f"Unique Paper 1 sequences: {len(p1_seqs)}")

# For each sequence, count MEROPS cleavage events matching its terminal AAs
results = []
for seq in sorted(p1_seqs):
    n_aa = seq[0]  # N-terminal AA (aminopeptidase target)
    c_aa = seq[-1]  # C-terminal AA (carboxypeptidase target)
    
    # How often does MEROPS see this AA at P1 (aminopeptidase cleavage)?
    n_cleavages = merops[merops["P1_1"] == n_aa].shape[0]
    
    # How often at P1' (carboxypeptidase -- the leaving group)?
    c_cleavages = merops[merops["P1p_1"] == c_aa].shape[0]
    
    # Top protease family for this N-terminal AA
    n_top_fam = merops[merops["P1_1"] == n_aa]["protease_family"].value_counts().head(1)
    n_top = f"{n_top_fam.index[0]}({n_top_fam.values[0]})" if len(n_top_fam) > 0 else "none"
    
    results.append({
        "sequence": seq,
        "N_term_AA": n_aa,
        "C_term_AA": c_aa,
        "MEROPS_N_cleavages": n_cleavages,
        "MEROPS_C_cleavages": c_cleavages,
        "top_N_family": n_top,
    })

xref = pd.DataFrame(results)
xref = xref.sort_values("MEROPS_N_cleavages", ascending=False)

print(f"\nMEROPS cross-reference for Paper 1 terminal AAs:")
print(f"(Higher cleavage count = more MEROPS evidence that this AA is a good exopeptidase substrate)")
print()
print(xref.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(xref))
width = 0.35
ax.bar([i - width/2 for i in x], xref["MEROPS_N_cleavages"], width, 
       label="N-terminal (aminopeptidase P1)", color="#1976d2", alpha=0.8)
ax.bar([i + width/2 for i in x], xref["MEROPS_C_cleavages"], width,
       label="C-terminal (carboxypeptidase P1')", color="#d32f2f", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(xref["sequence"], rotation=90, fontsize=7)
ax.set_ylabel("MEROPS Cleavage Count")
ax.set_title("MEROPS Evidence for Terminal Cleavage of Paper 1 Sequences")
ax.legend()
plt.tight_layout()
plt.show()

### SAM'S EXPERTISE NEEDED

**Does MEROPS predict the right ranking for your peptides?**

The bar chart shows how much MEROPS "evidence" exists for cleavage at each sequence's terminal amino acids. Sequences with tall bars should (in theory) be more susceptible to exopeptidase degradation.

- Does the ranking match what you observed experimentally?
- Are there sequences where MEROPS says "high risk" but they were actually stable (or vice versa)?
- Note: MEROPS only considers the terminal AA identity, not the modification. Your mods (Ac, betaA) block the enzyme entirely.

Notes: 

---
## 7. ESM-2 Embedding Visualization

ESM-2 (facebook/esm2_t12_35M_UR50D) embeddings for the 19 unique Paper 1 variable-residue sequences. These capture learned protein structure patterns. Let's see if they separate in meaningful ways.

In [ ]:
with open("data/processed/esm2_embeddings.pkl", "rb") as f:
    emb_data = pickle.load(f)

sequences = emb_data["sequences"]
mean_emb = np.array(emb_data["mean"])

print(f"ESM-2 model: {emb_data['model']}")
print(f"Sequences: {len(sequences)}")
print(f"Embedding dim: {mean_emb.shape[1]}")
print(f"\nSequences: {sequences}")

# PCA to 2D
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(mean_emb)
print(f"\nPCA explained variance: {pca.explained_variance_ratio_[0]:.1%}, {pca.explained_variance_ratio_[1]:.1%}")

# Map sequences to their variable residue (last character of RGEFV-X)
var_residues = [seq[-1] for seq in sequences]

# Color by amino acid property
aa_property = {}
for aa in "ACDEFGHIKLMNPQRSTVWY":
    if aa in "AVILMFWP":
        aa_property[aa] = "Hydrophobic"
    elif aa in "STNQ":
        aa_property[aa] = "Polar"
    elif aa in "KRH":
        aa_property[aa] = "Positive"
    elif aa in "DE":
        aa_property[aa] = "Negative"
    else:
        aa_property[aa] = "Other"

colors_map = {"Hydrophobic": "#1976d2", "Polar": "#43a047", "Positive": "#d32f2f", 
              "Negative": "#ff9800", "Other": "#9e9e9e"}

fig, ax = plt.subplots(figsize=(10, 8))
for i, (seq, vr) in enumerate(zip(sequences, var_residues)):
    prop = aa_property.get(vr, "Other")
    color = colors_map[prop]
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c=color, s=100, zorder=3)
    ax.annotate(f"{seq}\n(X={vr})", (emb_2d[i, 0], emb_2d[i, 1]), 
                fontsize=7, ha="center", va="bottom", xytext=(0, 5),
                textcoords="offset points")

# Legend
for prop, color in colors_map.items():
    ax.scatter([], [], c=color, s=100, label=prop)
ax.legend(title="Variable Residue Type", loc="best")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("ESM-2 Embeddings of Paper 1 Sequences (PCA)")
plt.tight_layout()
plt.show()

print("\nNote: Only 19 sequences -- too few for ESM-2 to really shine.")
print("When we have embeddings for all 618 (or your 80K), this will be much more informative.")

---
## 8. Placeholder: YOUR 80K DATA

When you're ready to add your full dataset, paste the file path below and run this cell to get basic stats. This is where everything changes -- real measurements instead of calibration model outputs.

In [ ]:
# ============================================================
# WHEN YOU ADD YOUR 80K DATA: update this path and run
# ============================================================
YOUR_DATA_PATH = None  # e.g., "data/rozans-80k-raw.csv"

if YOUR_DATA_PATH is not None and os.path.exists(YOUR_DATA_PATH):
    your_data = pd.read_csv(YOUR_DATA_PATH)
    print(f"Loaded: {YOUR_DATA_PATH}")
    print(f"Shape: {your_data.shape}")
    print(f"\nColumns: {list(your_data.columns)}")
    print(f"\nFirst 5 rows:")
    display(your_data.head())
    print(f"\nBasic stats:")
    print(your_data.describe().to_string())
    
    # Check for sequence column
    seq_col = None
    for candidate in ["sequence", "seq", "peptide", "Sequence", "clean_sequence"]:
        if candidate in your_data.columns:
            seq_col = candidate
            break
    
    if seq_col:
        print(f"\nSequence column: '{seq_col}'")
        print(f"Unique sequences: {your_data[seq_col].nunique()}")
        print(f"Sequence lengths: {your_data[seq_col].str.len().describe()}")
        
        # Overlap with existing 618
        overlap = set(your_data[seq_col]) & set(rozans["clean_sequence"])
        print(f"\nOverlap with existing 618 peptides: {len(overlap)} sequences")
        print(f"New sequences: {your_data[seq_col].nunique() - len(overlap)}")
else:
    print("No data path set yet. Update YOUR_DATA_PATH above when ready.")